# Autocorrelação espacial --- prática

**Curso de Econometria Espacial**  
Prof. Pedro Amaral · Cedeplar/UFMG  

Dados: *chicagoSDOH* (census tracts de Chicago, determinantes sociais de saúde) via `libpysal.examples`.


## 1. Importações

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import libpysal
import esda

## 2. Dados: chicagoSDOH

In [ ]:
# Carrega o exemplo e lista os arquivos
sdoh = libpysal.examples.load_example('chicagoSDOH')
sdoh.get_file_list()

In [ ]:
# Lê o shapefile de census tracts
gdf = gpd.read_file(libpysal.examples.get_path('Chi-SDOH.shp'))
print(gdf.shape)
print('CRS:', gdf.crs)
gdf[['TRACTCE10', 'ChldPvt14', 'Blk', 'Pov14']].head()

O CRS é **EPSG:3435** (NAD83 / Illinois East, em pés) — já projetado.\n
Para contiguidade (Rainha) a projeção não é obrigatória, mas está adequada.


## 3. Matriz de pesos Rainha

In [ ]:
w_queen = libpysal.weights.Queen.from_dataframe(gdf, use_index=False)
print(w_queen.n, 'observações')
print('Ilhas:', w_queen.islands)

## 4. Grafo de vizinhança

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
gdf.plot(ax=ax, facecolor='lightgray', edgecolor='white', linewidth=0.3)
w_queen.plot(gdf, ax=ax,
             edge_kws=dict(color='firebrick', linewidth=0.25, alpha=0.5),
             node_kws=dict(marker=''))
ax.set_title('Grafo de vizinhança — Rainha (census tracts)')
ax.set_axis_off()
plt.tight_layout()
plt.show()


## 5. Histograma do número de vizinhos

In [ ]:
from matplotlib.ticker import MaxNLocator

card = pd.Series(w_queen.cardinalities, name='n_vizinhos')

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(card, bins=range(int(card.min()), int(card.max()) + 2),
             discrete=True, color='steelblue', edgecolor='white', ax=ax)
ax.set_xlabel('Número de vizinhos')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição do número de vizinhos — Rainha')
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

print(card.describe())


## 6. Variável binária e mapa

Usamos `Blk`: indicador de tract com maioria negra (0/1).


In [ ]:
print(gdf['Blk'].value_counts())

fig, ax = plt.subplots(figsize=(5, 5))
gdf.plot(column='Blk', ax=ax, categorical=True,
         cmap='Set2_r', edgecolor='white', linewidth=0.2,
         legend=True,
         legend_kwds={'title': 'Blk', 'loc': 'lower left'})
ax.set_title('Tracts com maioria negra (Blk = 1)')
ax.set_axis_off()
plt.tight_layout()
plt.show()


## 7. Join Count

In [ ]:
# Join Count exige W binária (não padronizada)
w_queen.transform = 'B'

jc = esda.join_counts.Join_Counts(gdf['Blk'].values, w_queen)

print(f"Transformação atual de W: '{w_queen.transform}'")
print(f"Número de unidades espaciais (N): {w_queen.n}")
print(f"Número total de conexões (Join total): {w_queen.s0}")
print(f'BB (1–1): {jc.bb:.0f}   p_sim = {jc.p_sim_bb:.4f}')
print(f'WW (0–0): {jc.ww:.0f}')
print(f'BW (1–0): {jc.bw:.0f}   p_sim = {jc.p_sim_bw:.4f}')
print(f'\nBB + WW + BW = {jc.bb + jc.ww + jc.bw:.0f}')
print(f'S0 / 2       = {w_queen.s0 / 2:.0f}')


A variável `Blk` indica se o census tract tem **maioria negra** (1) ou não (0). Com a matriz Rainha binária:

| Estatística | Valor | $p$-valor (permutação) |
|-------------|------:|------------------------:|
| BB (1–1)    | 846   | 0,001 |
| WW (0–0)    | 1.505 | — |
| BW (1–0)    | 242   | 1,000 |
| Total de junções | 2.593 | $= S_0/2$ |

**O que isso diz**

1. **BB significativo** ($p = 0{,}001$): há bem mais fronteiras entre tracts majoritariamente negros do que se esperaria sob aleatoriedade espacial. Ou seja, tracts com `Blk = 1` se aglomeram no mapa (autocorrelação espacial positiva).

2. **BW não indica dissimilaridade**: o $p = 1{,}000$ no teste de BW significa que não há excesso de junções mistas (1 ao lado de 0). Na verdade, 242 é um valor *baixo* em relação às permutações — o que é o outro lado da mesma moeda da segregação espacial.

3. **WW** é o tipo de junção mais frequente (1.505), o que faz sentido: a maioria dos tracts não é majoritariamente negra, então a maior parte das fronteiras é 0–0.

4. **Checagem**: $BB + WW + BW = S_0/2 = 2.593$. As contas fecham.

**Em uma frase:** o padrão de `Blk` em Chicago é de aglomeração espacial, não de misturas — tracts majoritariamente negros tendem a ser vizinhos de outros tracts majoritariamente negros.

## 8. Autocorrelação Espacial

In [ ]:
w_queen.transform = 'R'
y = gdf['ChldPvt14'].values
wy = libpysal.weights.lag_spatial(w_queen, y)

fig, ax = plt.subplots(figsize=(7, 6))
sns.regplot(x=y, y=wy, ax=ax,
            scatter_kws=dict(s=25, color='steelblue'),
            line_kws=dict(color='firebrick', linewidth=2))
ax.set_xlabel(r'ChldPvt14 ($y$)')
ax.set_ylabel(r'W_ChldPvt14 ($Wy$)')
ax.set_title('ChldPvt14 vs. média dos vizinhos (Rainha)')
plt.tight_layout()
plt.show()


## 9. $I$ de Moran Global calculado manualmente

Com $W$ padronizada em linha, $S_0 = N$ e a fórmula simplifica para:
$$
I = \frac{\mathbf{z}' W \mathbf{z}}{\mathbf{z}' \mathbf{z}}
$$


In [ ]:
varname = 'ChldPvt14'
variable = gdf[varname]
variable_z = variable - variable.mean()
variable_array = variable_z.to_numpy()

# Defasagem dos desvios (W já está 'R')
lag_z = libpysal.weights.lag_spatial(w_queen, variable_array)

moran = np.dot(variable_array, lag_z) / np.dot(variable_array, variable_array)
print(f"Global Moran's I statistic: {moran:.4f}")


## 10. $p$-valor por permutação (manual)

In [ ]:
n_permutations = 99
simulated_morans = []
denominator = np.dot(variable_array, variable_array)

rng = np.random.default_rng(seed=123) 
for i in range(n_permutations):
    shuffled_z = rng.permutation(variable_array)
    lag_shuffled = libpysal.weights.lag_spatial(w_queen, shuffled_z)
    sim_i = np.dot(shuffled_z, lag_shuffled) / denominator
    simulated_morans.append(sim_i)

p_value = (np.sum(np.array(simulated_morans) >= moran) + 1) / (n_permutations + 1)

print(f"Observed Moran's I: {moran:.4f}")
print(f"P-value: {p_value:.4f}")

# Visualização da distribuição empírica
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(simulated_morans, bins=20, color='steelblue', edgecolor='white', ax=ax)
ax.axvline(moran, color='firebrick', linewidth=2, label=f'I observado = {moran:.3f}')
ax.set_xlabel("I de Moran (permutações)")
ax.set_ylabel('Frequência')
ax.set_title('Distribuição empírica sob H0 (aleatoriedade espacial)')
ax.legend()
plt.tight_layout()
plt.show()


## 11. Mesmo cálculo via `esda`

In [ ]:
mi = esda.moran.Moran(gdf['ChldPvt14'], w_queen, permutations=99)

print(f"I de Moran:  {mi.I:.4f}")
print(f"E[I]:        {mi.EI:.4f}")
print(f"p_sim:       {mi.p_sim:.4f}")
print(f"z_norm:      {mi.z_norm:.3f}")
print(f"p_norm:      {mi.p_norm:.4f}")


- **$I = 0{,}55$** está bem acima de $\mathrm{E}[I] \approx 0$: tracts com pobreza infantil parecida tendem a ser vizinhos (aglomeração de valores altos com altos e/ou baixos com baixos).

- **$p_{\mathrm{sim}} = 0{,}01$** é o menor $p$ possível com 99 permutações — $\frac{0+1}{99+1}$. Nenhuma das 99 permutações gerou $I$ tão alto quanto o observado. Com mais permutações (ex.: 999) o $p$ poderia cair ainda mais; o que importa é que rejeitamos $H_0$ com folga.

- **$z_{\mathrm{norm}} \approx 27{,}5$** confirma o mesmo pelo caminho analítico (aproximação normal): o resultado está longe da cauda da distribuição nula.

- O $I$ global **não diz onde** estão os aglomerados nem se são de pobreza alta ou baixa — isso fica a cargo do LISA.

Ou seja, a pobreza infantil nos census tracts de Chicago é espacialmente concentrada; a hipótese de aleatoriedade espacial é rejeitada.

## 12. $I$ de Moran Local — ChldPvt14

Quadrantes: HH (alto–alto), LL (baixo–baixo), HL (alto–baixo), LH (baixo–alto).


In [ ]:
lmi = esda.moran.Moran_Local(gdf['ChldPvt14'], w_queen, permutations=99, seed=123)

lmi.plot_scatter()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
lmi.plot(gdf, ax=ax, legend=True, edgecolor='darkgrey', linewidth=0.5)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Contagem por categoria LISA
cluster = lmi.q.copy()
cluster[lmi.p_sim >= 0.05] = 0

labels = {0: 'Não significativo', 1: 'HH', 2: 'LH', 3: 'LL', 4: 'HL'}

tabela_lisa = (
    pd.Series(cluster)
    .map(labels)
    .value_counts()
    .rename_axis('Categoria')
    .reset_index(name='N')
)
tabela_lisa['%'] = (tabela_lisa['N'] / tabela_lisa['N'].sum() * 100).round(1)
tabela_lisa

## Exercício

Usando os arquivos `RMBH.shp` (recorte territorial da RMBH) e `pratica.csv` (dados):

1. Utilizando uma matriz de pesos Rainha (Queen), calcule o I de Moran Global da variável 'pop';
2. Plote o scatterplot e o cluster map do I de Moran Local;
3. Interprete os resultados.

In [ ]:
# Carrega o território e os dados da prática
rmbh = gpd.read_file('RMBH.shp')
dados = pd.read_csv('pratica.csv')

print('Shapefile:', rmbh.shape, '| CRS:', rmbh.crs)
print('Dados:', dados.shape)
print('Colunas shp:', list(rmbh.columns))
print('Colunas csv:', list(dados.columns))

In [ ]:
# Merge: cod6 (shp) com cod (csv)
gdf_ex = rmbh.merge(dados, left_on='COD6', right_on='cod', how='inner')
print('Após merge:', gdf_ex.shape)
gdf_ex.head()